# Food calorie & macro estimation (OpenAI vision)

Uses [`MacrosEstimator`](food_classifier.py) from `food_classifier.py`.

```python
result = estimator.estimate_macros(
    note="...",
    photos=["path/to/image.jpg"],
)
result.model_dump()
```

At least one of `note` or `photos` is required.

**Constructor params** (all set explicitly in the setup cell below):

| Param | Default | Purpose |
|-------|---------|---------|
| `client` | from `OPENAI_API_KEY` | OpenAI client; `None` builds one automatically |
| `text_model` | `gpt-5-nano` | Model for note-only requests |
| `image_model` | `gpt-5-mini` | Model when any photo is attached |
| `kj_to_kcal_factor` | `4.184` | kJ → kcal conversion for label reads |
| `max_image_edge` | `1024` | Longest side (px) before JPEG resize |
| `jpeg_quality` | `85` | JPEG quality for API upload |
| `max_retries` | `3` | OpenAI client retries |
| `load_env` | `True` | Load `.env` on init |

Env overrides: `OPENAI_FOOD_TEXT_MODEL`, `OPENAI_FOOD_IMAGE_MODEL`, `OPENAI_FOOD_MODEL` (legacy image fallback).

**Model routing**
- **Note/description only** → `text_model`
- **Any photo attached** → `image_model`

The model used is returned on `result.model`.

**Supported photo formats:** `.jpg`, `.jpeg`, `.png`, `.heic`, `.heif` (all resized and converted to JPEG before sending).

**Energy handling**
- **Label photos:** the model reads raw per-serving energy (kJ or kcal); Python converts kJ → kcal using `kj_to_kcal_factor`.
- **Estimates:** the model returns kcal directly.

**Examples**
- Biscuits: `{ "note": "I ate 4 biscuits", "photos": ["label.jpg"] }`
- iPhone photo: `{ "note": "I ate 4 biscuits", "photos": ["IMG_1234.HEIC"] }`
- Dinner: `{ "note": "Grilled salmon, potatoes, ate everything", "photos": ["plate.jpg"] }`
- Text only: `{ "note": "2 eggs and toast with butter" }`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
FIXTURES = ROOT / "tests" / "fixtures" / "images"

In [1]:
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from food_classifier import MacrosEstimator
from food_models import FoodEstimateResult

# --- MacrosEstimator params (match food_classifier.py defaults) ---
LOAD_ENV = True
if LOAD_ENV:
    load_dotenv(ROOT / ".env", override=True)

def _model_from_env(key: str, default: str) -> str:
    return (os.environ.get(key) or default).strip()

CLIENT = None
TEXT_MODEL = _model_from_env("OPENAI_FOOD_TEXT_MODEL", "gpt-5-nano")
IMAGE_MODEL = (
    _model_from_env("OPENAI_FOOD_IMAGE_MODEL", "")
    or _model_from_env("OPENAI_FOOD_MODEL", "")
    or "gpt-5-mini"
)
KJ_TO_KCAL_FACTOR = 4.184
MAX_IMAGE_EDGE = 1024
JPEG_QUALITY = 85
MAX_RETRIES = 3

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

estimator = MacrosEstimator(
    client=CLIENT,
    text_model=TEXT_MODEL,
    image_model=IMAGE_MODEL,
    kj_to_kcal_factor=KJ_TO_KCAL_FACTOR,
    max_image_edge=MAX_IMAGE_EDGE,
    jpeg_quality=JPEG_QUALITY,
    max_retries=MAX_RETRIES,
    load_env=LOAD_ENV,
)

print(f"text_model={estimator.text_model!r}, image_model={estimator.image_model!r}")

text_model='gpt-5-nano', image_model='gpt-5-mini'


In [2]:
def show_estimate(result: FoodEstimateResult) -> None:
    out = result.output
    macros = out.macros_g

    kj_factor = estimator.kj_to_kcal_factor
    energy_line = f"- **Calories:** {out.calories_kcal} kcal"
    if out.energy_raw and out.energy_raw.unit == "kJ":
        raw = out.energy_raw
        energy_line = (
            f"- **Calories:** {out.calories_kcal} kcal "
            f"(from {raw.value:g} kJ ÷ {kj_factor:g})"
        )

    meta_lines = [
        f"- **Macros:** P {macros.protein}g | C {macros.carbs}g | F {macros.fat}g",
        f"- **Confidence:** {out.confidence} ({out.source})",
        f"- **Model:** {result.model}",
    ]
    if result.usage:
        meta_lines.append(f"- **Tokens:** {result.usage.total_tokens}")

    display(
        Markdown(
            f"""### {out.summary}

{energy_line}
"""
            + "\n".join(meta_lines)
        )
    )

    if out.assumptions:
        assumptions = "\n".join(f"- {item}" for item in out.assumptions)
        display(Markdown(f"**Assumptions**\n{assumptions}"))

    display(result.model_dump())

## Example: text only

No photo — the model estimates from typical values.

In [3]:
result = estimator.estimate_macros(
    note="I ate 4 digestive biscuits. 16g of carb in each biscuit",
    photos=None,
)
show_estimate(result)

### Estimated four digestive biscuits with 16 g carbs each; total carbs 64 g, assumed 1 g protein and 3 g fat per biscuit, totaling 380 kcal.

- **Calories:** 380 kcal
- **Macros:** P 4.0g | C 64.0g | F 12.0g
- **Confidence:** low (estimate)
- **Model:** gpt-5-nano
- **Tokens:** 3031

**Assumptions**
- No nutrition label available; estimate derived from stated carbs per biscuit.
- Each biscuit contains 1 g protein and 3 g fat (per biscuit) to match common biscuit nutrition ranges.
- Total servings consumed = 4 biscuits, so multiply per-biscuit macros by 4.
- Calories computed as: carbs 64 g × 4 kcal/g + fat 12 g × 9 kcal/g + protein 4 g × 4 kcal/g.

{'input': {'note': 'I ate 4 digestive biscuits. 16g of carb in each biscuit',
  'photos': []},
 'output': {'label_energy': None,
  'calories_kcal': 380,
  'macros_g': {'protein': 4.0, 'carbs': 64.0, 'fat': 12.0},
  'confidence': 'low',
  'source': 'estimate',
  'summary': 'Estimated four digestive biscuits with 16 g carbs each; total carbs 64 g, assumed 1 g protein and 3 g fat per biscuit, totaling 380 kcal.',
  'assumptions': ['No nutrition label available; estimate derived from stated carbs per biscuit.',
   'Each biscuit contains 1 g protein and 3 g fat (per biscuit) to match common biscuit nutrition ranges.',
   'Total servings consumed = 4 biscuits, so multiply per-biscuit macros by 4.',
   'Calories computed as: carbs 64 g × 4 kcal/g + fat 12 g × 9 kcal/g + protein 4 g × 4 kcal/g.'],
  'energy_raw': None},
 'model': 'gpt-5-nano',
 'usage': {'prompt_tokens': 415,
  'completion_tokens': 2616,
  'total_tokens': 3031}}

## Example: nutrition label + quantity

Point `photos` at a label image on disk. Add your own file under `your_data/food/` or anywhere else.

In [4]:
LABEL_PHOTO = FIXTURES / "brownies.HEIC"

if LABEL_PHOTO.is_file():
    result = estimator.estimate_macros(
        note="I ate 2 brownies",
        photos=[LABEL_PHOTO],
    )
    show_estimate(result)
else:
    print(f"Add a label photo at {LABEL_PHOTO} to run this example.")

### Per the product label (885 kJ per 45 g serving = one brownie) scaled to 2 brownies.

- **Calories:** 423 kcal (from 1770 kJ ÷ 4.184)
- **Macros:** P 4.4g | C 52.2g | F 21.0g
- **Confidence:** high (label)
- **Model:** gpt-5-mini
- **Tokens:** 2456

{'input': {'note': 'I ate 2 brownies', 'photos': ['brownies.HEIC']},
 'output': {'label_energy': {'per_serving': 885.0,
   'unit': 'kJ',
   'servings_consumed': 2.0},
  'calories_kcal': 423,
  'macros_g': {'protein': 4.4, 'carbs': 52.2, 'fat': 21.0},
  'confidence': 'high',
  'source': 'label',
  'summary': 'Per the product label (885 kJ per 45 g serving = one brownie) scaled to 2 brownies.',
  'assumptions': [],
  'energy_raw': {'value': 1770.0, 'unit': 'kJ'}},
 'model': 'gpt-5-mini',
 'usage': {'prompt_tokens': 1328,
  'completion_tokens': 1128,
  'total_tokens': 2456}}

## Example: dinner photo + description

In [5]:
MEAL_PHOTO = FIXTURES / "boerewors_roll.HEIC"

if MEAL_PHOTO.is_file():
    result = estimator.estimate_macros(
        note="2 boerewors rolls. pick n pay fresh hot dog rolls. a bit of carmelised onion and tomato sauce. also hot chips with some tomato sauce",
        photos=[MEAL_PHOTO],
    )
    show_estimate(result)
else:
    print(f"Add a meal photo at {MEAL_PHOTO} to run this example.")

### Estimated by summing two boerewors sausages, two hot dog rolls, a small amount of caramelised onion and ketchup, and a portion of hot chips using typical portion sizes and nutrient values.

- **Calories:** 1370 kcal
- **Macros:** P 44.0g | C 124.0g | F 76.0g
- **Confidence:** medium (estimate)
- **Model:** gpt-5-mini
- **Tokens:** 2703

**Assumptions**
- Two boerewors sausages total ~200 g cooked (≈300 kcal/100 g): 600 kcal, 28 g protein, 50 g fat, ~2 g carbs.
- Two Pick n Pay fresh hot dog rolls ≈120 g total (≈265 kcal/100 g): 318 kcal, 10.8 g protein, 58.8 g carbs, 4.2 g fat.
- Hot chips portion ≈120 g (fried fries ~312 kcal/100 g): 374 kcal, 4.1 g protein, 49 g carbs, 18 g fat.
- Caramelised onion ≈30 g including some oil: 50 kcal, ~0.3 g protein, 6 g carbs, 4 g fat.
- Tomato sauce/ketchup total ≈30 g (≈100 kcal/100 g): 30 kcal, ~0.3 g protein, 8 g carbs, 0 g fat.
- All ingredient weights and nutrient densities are estimated from typical values; photo used to judge relative portion sizes but exact weights are unknown.

{'input': {'note': '2 boerewors rolls. pick n pay fresh hot dog rolls. a bit of carmelised onion and tomato sauce. also hot chips with some tomato sauce',
  'photos': ['/Users/andrewallkin/personal/repos/garmin-api/boerewors_roll.HEIC']},
 'output': {'label_energy': None,
  'calories_kcal': 1370,
  'macros_g': {'protein': 44.0, 'carbs': 124.0, 'fat': 76.0},
  'confidence': 'medium',
  'source': 'estimate',
  'summary': 'Estimated by summing two boerewors sausages, two hot dog rolls, a small amount of caramelised onion and ketchup, and a portion of hot chips using typical portion sizes and nutrient values.',
  'assumptions': ['Two boerewors sausages total ~200 g cooked (≈300 kcal/100 g): 600 kcal, 28 g protein, 50 g fat, ~2 g carbs.',
   'Two Pick n Pay fresh hot dog rolls ≈120 g total (≈265 kcal/100 g): 318 kcal, 10.8 g protein, 58.8 g carbs, 4.2 g fat.',
   'Hot chips portion ≈120 g (fried fries ~312 kcal/100 g): 374 kcal, 4.1 g protein, 49 g carbs, 18 g fat.',
   'Caramelised onion ≈

## Your log entry

Edit `note` and `photos` below for whatever you ate.

In [ ]:
note = ""
photos: list[str | Path] = []

# --- examples (uncomment one) ---
# note = "I ate 4 biscuits"
# photos = ["your_data/food/label.jpg"]

# note = "Chicken stir-fry with rice, ate everything"
# photos = ["your_data/food/plate.jpg"]

if note.strip() or photos:
    result = estimator.estimate_macros(
        note=note or None,
        photos=photos or None,
    )
    show_estimate(result)
else:
    print("Set note and/or photos above, then re-run this cell.")